In [ ]:
#Install a few packages that aren’t automatically included in Colab
!pip install windrose cartopy cmocean

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 15.4 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import warnings
from datetime import datetime, timedelta
from matplotlib import pyplot as plt
import numpy as np
from windrose import WindroseAxes
import math
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import xarray as xr
import cmocean
import calendar
import geopy.distance
from matplotlib.colors import ListedColormap
import seaborn as sns
import statsmodels.api as sm
import sklearn.metrics as skmet
import matplotlib.patches as mpatches
import re
from google.colab import drive

%matplotlib inline
warnings.filterwarnings("ignore")

drive.mount('/content/drive')

main_dir = '/content/drive/MyDrive/TOR_ETC_Analysis' #path to main location to store data and plots **NOTE: add shortcut from shared folder to your MyDrive
data_dir = os.path.join(main_dir, 'data')
plots_dir = os.path.join(main_dir, 'plots')


Mounted at /content/drive


In [ ]:
!ls "/content/drive/My Drive/TOR_ETC_Analysis" # make sure main_dir is present and accessible

Analysis.ipynb	data  Data_Processing.ipynb  Plot_Hourly_Events.ipynb  plots


# Plot tornadoes, ETCs, and environment

In [ ]:
###############################
# Plot all data for hourly data
###############################
def plotData(req_df, T2M, SP, SH, date, date_obj):
    plt.ioff()

    # Select correct data points
    t2m_hour_select = T2M.sel(time=date)
    sh_hour_select = SH.sel(time=date)
    sp_hour_select = SP.sel(time=date)

    # Initialize figure, set size and resolution
    fig = plt.figure(figsize=(12, 10), dpi=150)
    fig.set_facecolor('white')

    # create axis using cartopy projections, add in coastlines & grid
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color='grey', zorder=2)
    ax.add_feature(cfeature.STATES, edgecolor='grey')
    ax.grid(visible=True)

    # create gridlines and format labels
    gl = ax.gridlines(draw_labels=True, zorder=3)
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabels_top = gl.ylabels_right = False
    gl.xlines = gl.ylines = True
    gl.xlabel_style = {'size': 14}
    gl.ylabel_style = {'size': 14}

    xlim = ([-135, -55])
    ylim = ([20, 70])
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # Plot filled contour of 2M Temp, including colorbar
    fc = (t2m_hour_select.VAR_2T[:, :]).plot.contourf(ax=ax, transform=ccrs.PlateCarree(), extend='both', add_colorbar=False,
                                                      levels=np.arange(250, 310, 5), cmap=cmocean.cm.thermal)
    cbar = fig.colorbar(fc, label='Near Surface (2m) Temperature [K]', shrink = 0.5)

    # Plot black contour lines of MSL - hPa
    fc1 = (sp_hour_select.MSL[:, :]/100).plot.contour(ax=ax, transform=ccrs.PlateCarree(), extend='both', add_colorbar=False,
                                                      levels=np.arange(990, 1030, 4), colors="white")  # , lw=2.0
    ax.clabel(fc1, inline=True, fontsize=10)

    # Plot black contour dashed lines of Q
    # Multiple by 1000 to convert from kg kg**-1 to g kg**-1
    fc2 = (sh_hour_select.Q[:, :]*1000).plot.contour(ax=ax, transform=ccrs.PlateCarree(), extend='both', add_colorbar=False,
                                                     levels=np.arange(0, 50, 3), colors="black", linestyles='dashed')
    ax.clabel(fc2, inline=True, fontsize=10)

    # Plot tornado data
    tor_df = pd.DataFrame(columns=['slat', 'slon'])
    tor_lats = []
    tor_lons = []

    # Store latitudes
    for lat in req_df['slat']:
        tor_lats.append(lat)

    # Store longitudes
    for lon in req_df['slon']:
        tor_lons.append(lon)

    tor_count = len(tor_lats)
    i = 0
    while i < tor_count:
        df = pd.DataFrame({"slat": [tor_lats[i]], "slon": [tor_lons[i]]})
        tor_df = tor_df.append(df, ignore_index=True)
        i = i + 1
    if not tor_df.empty:
        ax.scatter(tor_df['slon'], tor_df['slat'], transform=ccrs.PlateCarree(),
                   color='blue', marker='v', s=100, zorder=5, edgecolors='darkblue')

    # Plot extratropical cyclone data
    ext_data = pd.DataFrame(columns=['etclats', 'etclons'])
    ext_lats = []
    ext_lons = []

    # Store latitudes
    for etc_lats in req_df['etclats']:
        # Fix data from str to list
        etc_lats = etc_lats.replace("[", "").replace("]", "")
        etc_lats = etc_lats.split(" ")
        for lat in etc_lats:
            if not lat == '':
                ext_lats.append(float(lat))

    # Store longitudes
    for etc_lons in req_df['etclons']:
        # Fix data from str to list
        etc_lons = etc_lons.replace("[", "").replace("]", "")
        etc_lons = etc_lons.split(" ")
        for lon in etc_lons:
            if not lon == '':
                ext_lons.append(float(lon))

    ext_count = len(ext_lats)
    i = 0
    while i < ext_count:
        df = pd.DataFrame({"etclats": [ext_lats[i]], "etclons": [ext_lons[i]]})
        ext_data = ext_data.append(df, ignore_index=True)
        i = i + 1
    if not ext_data.empty:
        ax.scatter(ext_data['etclons'], ext_data['etclats'], transform=ccrs.PlateCarree(),
                   color='green', marker="o", s=150, zorder=5, edgecolors='darkgreen')

    plt.title(r" %s  tornado count: %s" % (str(date_obj), tor_count))

    year_save_dir = '%s/%s' % (plots_dir, str(date_obj.year))
    save_dir = '%s/%s' % (year_save_dir, str(date_obj.month).zfill(2))

    if not os.path.exists(year_save_dir):
        os.mkdir(year_save_dir)
    if os.path.exists(save_dir):
        plt.savefig("%s/tornado_%s.png" % (save_dir, str(date_obj)), bbox_inches='tight')
    else:
        os.mkdir(save_dir)
        plt.savefig("%s/tornado_%s.png" % (save_dir, str(date_obj)), bbox_inches='tight')

    plt.show()
    plt.close()
    plt.clf()


###############################
# Store data to be plotted
###############################
def storeWeatherData(req_df, date_hr):
    if not str(date_hr) == "nan":
        dt_obj = datetime.strptime(str(date_hr), '%Y-%m-%d %H:%M:%S')
        date = str(dt_obj.strftime("%Y-%m-%d"))
        year = str(dt_obj.year)
        month = str(dt_obj.month).zfill(2)
        day = str(dt_obj.day).zfill(2)
        year_month = year + month
        print(year_month)
        N_days = calendar.monthrange(int(year), int(month))[1]
        print(N_days)

        # Files found on the RDA Web Server: https://data.rda.ucar.edu/ds633.0    NOTE: this will change on Jul 30 - the 'ds633.0' will become 'd633000'
        # or if accessing on UCARs Jupyter notebook: /glade/campaign/collections/rda/data/ds633.0
        access = 'https://data.rda.ucar.edu/ds633000'

        # 2 Metre Temperature units K
        code = '167'
        var = '2t'
        oper = 'e5.oper.an.sfc'
        url = f'{access}/{oper}/{year_month}/{oper}.128_{code}_{var}.ll025sc.{year_month}0100_{year_month}{N_days}23.nc'
        T2M = xr.open_dataset(url).sel(latitude=slice(70.0, 20.0), longitude=slice(150.0, 310.0), time=date).load()

        # Specific Humidity units kg/kg
        code = '133'
        var = 'q'
        oper = 'e5.oper.an.pl'
        plvl = '1000.'  # check units - hPa
        url = f'{access}/{oper}/{year_month}/{oper}.128_{code}_{var}.ll025sc.{year_month}{day}00_{year_month}{day}23.nc'
        SH = xr.open_dataset(url).sel(latitude=slice(70.0, 20.0), longitude=slice(150.0, 310.0), time=date, level=plvl).load()

        # Mean Sea-Level Pressure units Pa
        code = '151'
        var = 'msl'
        oper = 'e5.oper.an.sfc'
        url = f'{access}/{oper}/{year_month}/{oper}.128_{code}_{var}.ll025sc.{year_month}0100_{year_month}{N_days}23.nc'
        SP = xr.open_dataset(url).sel(latitude=slice(70.0, 20.0), longitude=slice(150.0, 310.0), time=date).load()

        plotData(req_df, T2M, SP, SH, date_hr, dt_obj)

In [ ]:
# Plot tornadoes, ETCs, and environment
# NOTE: This takes a very long time (days) to run for all years
# Recommend plotting the year (month, day, hour, etc) needed
tor_file = '%s/1980-2022_add_non_tc_tor_and_etcs.csv' % data_dir
if os.path.exists(tor_file):
    tor_data = pd.read_csv(tor_file, header=[0])
    for date_times in tor_data['utcdatetimes'].unique():
        # To only plot a specific year, uncomment next two lines
        # year = date_times[:4]
        # if year == "1982":
        data = tor_data[tor_data['utcdatetimes'].isin([date_times])]
        storeWeatherData(data, date_times)

198001
31


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'https://thredds.rda.ucar.edu/thredds/dodsC/files/g/ds633.0/e5.oper.an.sfc/198001/e5.oper.an.sfc.128_167_2t.ll025sc.1980010100_1980013123.nc', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)